# 🎲 Assignment 9 — Evaluation of OS Random Number Generation Mechanisms
### Course: Cryptography Laboratory (22CPP307)
### Group: 01 | Nishant (2024UCP1773) & Lokesh Saini (2024UCP1505)

---

## Executive Summary & Objectives
This notebook evaluates operating-system random-number generation mechanisms and investigates their suitability for cryptographic applications:
1. **Empirical Evaluation of 1 MB Datasets**:
   - Linux `/dev/urandom` (ChaCha20-based kernel stream)
   - OpenSSL `rand` (NIST SP 800-90A DRBG)
   - Windows CNG (`BCryptGenRandom` via Cryptography Next Generation)
   - Python PRNG (Control group: non-cryptographic Mersenne Twister MT19937)
2. **Statistical Analysis**: Byte frequencies, Chi-Square goodness-of-fit, Monobit test, Runs test, Autocorrelation, Shannon Entropy, and Monte Carlo estimation of $\pi$.
3. **OS Architecture Deep-Dive**: How Linux and Windows harvest physical entropy and manage internal CSPRNG pools.
4. **Fundamental Cryptographic Question**: Why passing statistical tests does **NOT** guarantee cryptographic security.


In [ ]:
import os
import sys
import math
import time
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Adjust path to import testing suite
sys.path.append(os.path.abspath('../src'))
from statistical_tests import (
    monobit_test, byte_frequency_test, runs_test,
    autocorrelation_test, shannon_entropy, monte_carlo_pi,
    evaluate_dataset
)

data_dir = os.path.abspath('../data')
print("Data directory:", data_dir)
datasets = {
    'Linux /dev/urandom': os.path.join(data_dir, 'dev_urandom_1mb.bin'),
    'OpenSSL rand': os.path.join(data_dir, 'openssl_1mb.bin'),
    'Windows CNG': os.path.join(data_dir, 'windows_cng_1mb.bin'),
    'Python PRNG (MT19937)': os.path.join(data_dir, 'python_prng_1mb.bin')
}

for name, path in datasets.items():
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f"{name:<25}: {size:>10,} bytes ({'OK' if size == 1048576 else 'MISSING'})")


---
## 1. Byte-Frequency Distribution and $\chi^2$ Goodness-of-Fit Test
For a truly uniform byte distribution, each byte value $0 \le v \le 255$ has an expected frequency of:
$$E = \frac{1,048,576}{256} = 4,096 \text{ occurrences}$$

The Chi-Square test statistic evaluates deviations:
$$\chi^2 = \sum_{i=0}^{255} \frac{(O_i - E)^2}{E}, \quad df = 255$$
For $df = 255$, acceptable range for $\alpha=0.01$ is roughly $[198.5, 319.4]$.


In [ ]:
results = {}
for name, path in datasets.items():
    results[name] = evaluate_dataset(path)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    counts = res['byte_frequency']['byte_counts']
    chi2 = res['byte_frequency']['chi2_statistic']
    pval = res['byte_frequency']['p_value']
    
    ax.bar(range(256), counts, color='royalblue', alpha=0.7, width=1.0)
    ax.axhline(4096, color='red', linestyle='--', linewidth=1.5, label='Expected: 4,096')
    ax.set_title(f"{name}\n$\chi^2={chi2:.2f}$, $p={pval:.4f}$ ({'PASS' if res['byte_frequency']['passed'] else 'FAIL'})")
    ax.set_xlabel("Byte Value (0..255)")
    ax.set_ylabel("Observed Count")
    ax.set_ylim(3700, 4500)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 2. Serial Autocorrelation (Lags 1 to 20)
Autocorrelation evaluates whether knowledge of byte $X_t$ reveals linear information about byte $X_{t+k}$:
$$r_k = \frac{\sum (X_i - \bar{X})(X_{i+k} - \bar{X})}{\sum (X_i - \bar{X})^2}$$

For independent random data of size $N = 1,048,576$:
- $95\%$ confidence bounds: $\pm \frac{1.96}{\sqrt{N}} \approx \pm 0.001914$
- $99\%$ confidence bounds: $\pm \frac{2.576}{\sqrt{N}} \approx \pm 0.002516$


In [ ]:
plt.figure(figsize=(14, 6))
lags = list(range(1, 21))
bound_95 = 1.96 / math.sqrt(1048576)
bound_99 = 2.576 / math.sqrt(1048576)

colors = {'Linux /dev/urandom': 'green', 'OpenSSL rand': 'purple', 'Windows CNG': 'blue', 'Python PRNG (MT19937)': 'orange'}

for name, path in datasets.items():
    with open(path, 'rb') as f:
        data = f.read()
    autocorr = autocorrelation_test(data, max_lag=20)
    coeffs = [autocorr['correlations'][k] for k in lags]
    plt.plot(lags, coeffs, marker='o', label=name, color=colors[name], alpha=0.8)

plt.axhline(0, color='black', linestyle='-', linewidth=0.8)
plt.axhline(bound_95, color='red', linestyle='--', label=f'95% Bound (+/-{bound_95:.4f})')
plt.axhline(-bound_95, color='red', linestyle='--')
plt.axhline(bound_99, color='darkred', linestyle=':', label=f'99% Bound (+/-{bound_99:.4f})')
plt.axhline(-bound_99, color='darkred', linestyle=':')

plt.title("Serial Autocorrelation vs Lag (1 to 20) Across All 4 Generators", fontsize=13)
plt.xlabel("Lag k (Bytes)", fontsize=11)
plt.ylabel("Autocorrelation Coefficient $r_k$", fontsize=11)
plt.xticks(lags)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower left')
plt.show()


---
## 3. Shannon Entropy & Monte Carlo Value for $\pi$
- **Shannon Entropy**: Measures the information content per byte. The maximum possible value is $\log_2(256) = 8.000000$ bits/byte.
- **Monte Carlo Estimation of $\pi$**: Treating pairs of bytes as 2D coordinates $(x, y) \in [0, 1)^2$, points satisfying $x^2 + y^2 \le 1.0$ approach $\pi / 4$.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entropy bar chart
names = list(results.keys())
entropies = [results[n]['entropy']['entropy'] for n in names]

axes[0].barh(names, entropies, color=['mediumseagreen', 'mediumorchid', 'cornflowerblue', 'sandybrown'])
axes[0].set_xlim(7.9995, 8.0000)
axes[0].axvline(8.0, color='red', linestyle='--', label='Theoretical Max (8.000000)')
axes[0].set_title("Shannon Entropy (bits/byte)")
axes[0].set_xlabel("Bits per Byte")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='lower right')

# Monte Carlo Pi comparison
pis = [results[n]['monte_carlo_pi']['estimated_pi'] for n in names]

axes[1].bar(names, pis, color=['mediumseagreen', 'mediumorchid', 'cornflowerblue', 'sandybrown'], width=0.5)
axes[1].axhline(math.pi, color='red', linestyle='--', label=f'True $\pi$: {math.pi:.5f}')
axes[1].set_ylim(3.13, 3.15)
axes[1].set_title("Monte Carlo Estimation of $\pi$")
axes[1].set_ylabel("Estimated Value")
axes[1].set_xticklabels(names, rotation=15, ha='right')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


---
## 4. Comprehensive Experimental Results Summary Table


In [ ]:
header = f"{'Generator / Mechanism':<24} | {'1s Ratio':<8} | {'Chi2':<8} | {'p-val':<6} | {'Entropy':<8} | {'Lag-1 r':<9} | {'Est. Pi':<8} | {'Status'}"
print(header)
print("-" * len(header))

for name, res in results.items():
    ones = res['monobit']['proportion_ones']
    chi2 = res['byte_frequency']['chi2_statistic']
    pval = res['byte_frequency']['p_value']
    ent = res['entropy']['entropy']
    lag1 = res['autocorrelation']['lag_1_correlation']
    pi_val = res['monte_carlo_pi']['estimated_pi']
    status = 'PASS' if res['all_passed'] else 'FAIL'
    print(f"{name:<24} | {ones:<8.5f} | {chi2:<8.2f} | {pval:<6.4f} | {ent:<8.6f} | {lag1:<+9.6f} | {pi_val:<8.5f} | {status}")


---
## 5. Architectural Investigation: How Linux and Windows Manage CSPRNGs (Task 4)

### 5.1 Linux Kernel CSPRNG Architecture
- **Entropy Harvesting**: The kernel continuously samples non-deterministic timing jitter from physical events:
  - Hardware device interrupts (`add_interrupt_randomness`)
  - Storage device and disk I/O completion times
  - Human interface interaction (keystroke and mouse timings)
  - CPU hardware random instructions (`RDRAND` / `RDSEED`)
- **Internal Mixing & DRBG**:
  - In modern kernels (Linux 4.8+, updated in 5.14 and 5.17+), entropy is accumulated into a primary pool using **BLAKE2s**.
  - Output is generated by per-CPU **ChaCha20 stream cipher** DRBGs.
  - **Fast Erasure / Forward Secrecy**: The ChaCha20 key is overwritten immediately after generating output blocks, preventing state reconstruction if the kernel memory is later compromised.
- **`/dev/urandom` vs `/dev/random`**:
  - Historically, `/dev/random` blocked when an entropy estimate counter reached zero, causing denial of service.
  - Modern Linux has unified both generators: once seeded at system boot with 256 bits of true entropy, **`/dev/urandom` is cryptographically secure and never blocks**.
  - The `getrandom(2)` syscall provides a clean userspace API that guarantees the pool has been seeded without path traversal vulnerabilities.

### 5.2 Windows CNG (`BCryptGenRandom`) Architecture
- **Evolution**: Windows migrated from the legacy `CryptoAPI` (`CryptGenRandom`) to `RtlGenRandom` (`advapi32.dll`), and finally to **Windows Cryptography Next Generation (CNG)** via `BCryptGenRandom` in `bcrypt.dll`.
- **Entropy Collection**:
  - High-precision performance counters (`QueryPerformanceCounter`)
  - Interrupt and DPC dispatch execution timestamps
  - ACPI power management hardware timers
  - Hardware TPM (Trusted Platform Module) entropy sources
- **Cryptographic Engine**:
  - Complies with **NIST SP 800-90A DRBG**.
  - Default implementation uses **AES-256 in Counter (CTR) mode** with continuous background reseeding.
  - Both Python on Windows (`os.urandom`) and .NET (`RandomNumberGenerator`) route directly to `BCryptGenRandom`.


---
## 6. The Core Cryptographic Question: Why Passing Statistical Tests Does NOT Guarantee Security (Task 5)

### 6.1 The Fundamental Distinction
- **Statistical Tests evaluate Output Distribution**: They check whether byte frequencies, 0/1 balances, and simple linear correlations match a memoryless uniform source.
- **Cryptographic Security evaluates Computational Unpredictability**: It demands that an adversary observing prior outputs cannot compute future outputs or recover earlier states.

### 6.2 Proof from our Experimental Data: The Python PRNG Paradox
Look closely at the table above:
- **`Python PRNG (Mersenne Twister MT19937)` passed EVERY single statistical test!**
  - Proportion of ones: $\approx 0.5000$
  - Chi-Square: $267.79$ ($p = 0.2787$)
  - Shannon Entropy: $7.999816$ bits/byte
  - Autocorrelation: perfectly within statistical confidence bounds
  - Monte Carlo $\pi$: $3.14197$

**Yet, Python's `random` module has ZERO cryptographic security!**
- The internal state of Mersenne Twister consists of exactly 624 32-bit integers ($19,937$ bits).
- By observing just 624 consecutive outputs, an attacker can invert the linear tempering transformations, recover the entire state matrix, and **predict every past and future output with 100% accuracy**.
- Using Python's default `random` for generating RSA keys, session tokens, or IVs leads to catastrophic vulnerabilities.

### 6.3 Other Classical Counterexamples
1. **The Binary Expansion of $\pi$**:
   - The digits of $\pi$ pass all NIST randomness tests (normal number properties, maximal entropy).
   - Yet $\pi$ is completely deterministic and publicly known to all adversaries.
2. **AES in Counter Mode with a Known Key**:
   - $C_i = \text{AES}_K(i)$.
   - Outputs have ideal statistical randomness, but anyone who knows or leaks $K$ can predict all outputs trivially.

### 6.4 Formal Requirements of a True CSPRNG
A generator can only be trusted for cryptography if it satisfies:
1. **Next-Bit Unpredictability (Yao's Test)**: Given $s_1, \dots, s_k$, no polynomial-time algorithm can predict $s_{k+1}$ with probability $> \frac{1}{2} + \text{negl}(k)$.
2. **Forward Secrecy (Backtracking Resistance)**: State compromise at time $t$ does not allow computing past outputs $s_{t-1}, s_{t-2}, \dots$.
3. **Backward Secrecy (Prediction Resistance)**: Fresh entropy injection ensures future outputs become unpredictable even after a temporary state compromise.
